# 00. Prepare the study data

This is the first of three notebooks. Here you build the dataset that notebooks 01 and 02 analyze.

## The causal question

> **To what extent does applying the backdoor defense, instead of baseline random filtering, change the probability of successful backdoor detection?**

## The words we will use

Causal inference has its own vocabulary. Four terms cover most of this notebook.

| Term | Plain meaning | Here |
|---|---|---|
| **Unit** | One thing you observe and could apply the treatment to. | One code example, which is one row of the table. |
| **Treatment** | The action whose effect you want to measure. This is the cause side of the question. | Applying the backdoor defense. |
| **Outcome** | The result you measure afterwards. This is the effect side. | Whether the pipeline detected the backdoor. |
| **Covariate** | Any other variable you record about a unit. Some matter causally, some do not. | Code complexity, reviewer experience, and so on. |

The word treatment comes from medicine, where it meant a drug. It does not have to be medical. It means whatever intervention you are studying, which here is a software defense.

In this dataset the treatment and the outcome both take the value 0 or 1.

| Variable | Meaning |
|---|---|
| `treatment = 0` | baseline random filtering |
| `treatment = 1` | backdoor defense applied |
| `outcome = 0` | detection failed |
| `outcome = 1` | detection succeeded |

Since `outcome` only ever takes the value 0 or 1, its average is the fraction of successes. We call this the detection success rate, or DSR. A DSR of 0.70 means the pipeline detected 70% of the examples.

**Tutorial path:** **00 Data preparation**, then 01 Correlational analysis, then 02 Causal inference


## 1. What is real and what is synthetic

The raw CSV holds real code and docstring examples. They give the dataset realistic variation from one unit to the next.

A program generates the treatment assignment and the detection outcome, so those two columns are synthetic. Treat the numbers as a teaching exercise rather than as evidence about any real backdoor defense.

Each row of the final table describes one code example under one treatment condition with one detection outcome.


## 2. Configure the preparation

Set the input file, the output path, and the random seed this notebook uses.

The seed fixes the random number generator, so everyone who runs this notebook builds the same dataset.


In [ ]:
from src.causal_data_prep import (
    DEFAULT_COVARIATES,
    engineer_features,
    load_source_data,
    make_synthetic_observational_data,
    save_causal_dataset,
    validate_causal_dataset,
)

def default_params():
    return {
        "source_dataset": "data/raw_code.csv",
        "lizard_cache_folder": "cache/lizard",
        "causal_dataset": "data/causal_data.csv",
        "random_seed": 42,
        "covariate_columns": DEFAULT_COVARIATES,
    }

params = default_params()
params


## 3. Load the source examples

Every variable in this tutorial comes from one of three places. Knowing where a variable comes from is what makes the timing argument possible later on.

The CSV holds 30,000 examples in five columns that the tutorial uses.

| Column | Type | Meaning |
|---|---|---|
| `input_code` | text | the code example itself |
| `output_docstring` | text | the docstring that documents it |
| `reviewer_experience` | number from 0 to 12 | how much experience the reviewer handling this example has |
| `rollout_eligibility` | 0 or 1 | whether policy has cleared this example for the new defense |
| `noise_feature` | number, average 0, spread 1 | a random number attached to each example, carrying no intended meaning |

### What rollout_eligibility represents

Teams rarely switch a new defense on everywhere at once. They roll it out gradually. Some part of the codebase gets cleared for it first, perhaps a pilot group or a few teams or repositories, while everything else keeps the old process for now.

The `rollout_eligibility` column records that administrative decision. It splits this dataset almost evenly, with about half the examples eligible.

Two properties make it interesting.

1. Policy sets it, not the example. Someone drew up a rollout plan, and the flag was already attached before the pipeline processed any of these examples.
2. It strongly influences which method an example receives. An eligible example is much more likely to get the defense, and an ineligible one mostly cannot.

That raises a question worth carrying with you. Does eligibility itself change the chance of detection, in any way other than by changing which method runs? Eligibility is a scheduling decision rather than a property of the code. Your answer decides whether you draw an arrow from `rollout_eligibility` to `outcome` in notebook 02, and it is a judgment about the process rather than something the correlations can settle for you.

### Why it matters that we know these first

All five columns hold their values before anyone decides which method to apply. A variable that already exists at decision time could have influenced that decision. A variable that comes into existence afterwards could not.


In [ ]:
source_df = load_source_data(params["source_dataset"])

print(f"Loaded {len(source_df):,} source examples.")
source_df[
    [
        "input_code",
        "output_docstring",
        "reviewer_experience",
        "rollout_eligibility",
        "noise_feature",
    ]
].head()


## 4. Engineer code features

The next four variables do not come from a file. This cell computes them by parsing `input_code`. They describe the code as someone wrote it, so like the columns above they hold their values before any treatment decision.

| Feature | How this cell computes it | What it captures |
|---|---|---|
| `code_number_tokens` | counts the tokens Python's own tokenizer produces, skipping whitespace and comments | how much code there is |
| `code_complexity` | cyclomatic complexity, from the `lizard` library | how many branching paths run through the code, since every `if`, loop, `and` or `or` adds one, and a function with no branches scores 1 |
| `code_num_identifiers` | counts the distinct names that are not Python keywords | how many separate things the code refers to |
| `code_num_strings` | counts string literals | how much text the code carries |

Parsing every example takes time, so this is the slow cell. The code caches each result, and a second run finishes quickly.

These four are measurements, meaning numbers that describe a unit. A measurement carries no causal role on its own. You decide whether `code_complexity` causes anything by reasoning about it in notebook 02. The column existing in the data settles nothing.

They also overlap with each other. Longer code tends to contain more identifiers and more branches. When several columns measure overlapping aspects of the same thing, some of them may turn out to be proxies, which carry no causal force themselves and merely stand in for something that does.


In [ ]:
feature_df = engineer_features(
    source_df,
    cache_dir=params["lizard_cache_folder"],
)

feature_df[
    [
        "code_number_tokens",
        "code_complexity",
        "code_num_identifiers",
        "code_num_strings",
        "reviewer_experience",
        "rollout_eligibility",
        "noise_feature",
    ]
].describe().T


## 5. Generate the observed treatment and outcome

Timing becomes concrete here. Picture each example moving through a pipeline, one stage at a time.

```text
  STAGE 1  the example arrives
           code features, reviewer_experience,
           rollout_eligibility, noise_feature        all of this already exists
                    |
                    v
  STAGE 2  a method is chosen                        treatment (0 or 1)
                    |
                    v
  STAGE 3  the chosen method runs                    inspection_intensity
                    |
                    v
  STAGE 4  it reports a verdict                      outcome (0 or 1)
                    |
                    v
  STAGE 5  the case may go to a human                manual_review_flag
```

Stage 2 assigns each unit to exactly one condition.

- `0` for random filtering, the baseline, which we also call the control condition.
- `1` for the backdoor defense, which we call the treated condition.

Nobody assigns the condition at random. Stage 1 characteristics influence which method an example receives, and that is what makes this an observational study rather than an experiment.

### Where the two later variables come from

Stages 3 and 5 produce two more variables:

`inspection_intensity` records how much inspection effort the chosen method actually spent on this example. It measures work that a method performed. Before stage 2 nobody has chosen a method, so no inspection has happened and nothing exists to measure. The number does not exist yet.

`manual_review_flag` records whether the case went to a human afterwards. That escalation happens once the automated pass finishes, so by the time anyone sets this flag, the method and the verdict are both already known. Either one could have influenced the decision to escalate.

So describing these two variables as measured after treatment is not a label somebody chose. It follows from how the pipeline runs. Turn it around and the point gets sharper. For the escalation flag to count as a variable known before treatment, someone would have to decide to escalate a case before choosing which method to run on it. The process does not work that way.

This matters because both variables correlate strongly with the treatment and with the outcome, which makes them tempting to adjust for. Notebook 02 shows why giving in to that temptation can make your answer worse rather than better.

### The fundamental problem

Each unit goes through the pipeline once. It receives one condition and produces one result, and we never see what would have happened to that same unit under the other condition. We call that unobserved alternative the counterfactual, and its absence is the central difficulty of causal inference. If we could see both results, we would simply subtract one from the other and finish.


In [ ]:
causal_df, study_info = make_synthetic_observational_data(
    feature_df,
    seed=params["random_seed"],
)

print(f"Backdoor-defense prevalence: {study_info.treatment_prevalence:.3f}")
print(f"Observed detection success rate: {study_info.outcome_prevalence:.3f}")

causal_df.head()


## 6. Validate the table

Check that:

- every unit appears exactly once;
- both treatment conditions appear, since with no control group you have nothing to compare against;
- `outcome` is binary, taking only the value 0 or 1;
- the analysis variables hold numeric, finite values;
- no analysis values are missing.


In [ ]:
validation_summary = validate_causal_dataset(
    causal_df,
    covariates=params["covariate_columns"],
)

validation_summary


## 7. Save the dataset

This cell writes `data/causal_data.csv`, the one file that notebooks 01 and 02 read.

It holds one row per code example, with the observed treatment, the observed outcome, and every covariate.


In [ ]:
data_path = save_causal_dataset(
    causal_df,
    params["causal_dataset"],
)

print(f"Saved causal dataset: {data_path}")


## Next: notebook 01

Notebook 00 answered one question: what did we observe for each unit?

Notebook 01 asks a different one: what relationships can we see in the observed data?

That is a question about correlation, not yet about cause. Keeping those two apart is the point of this tutorial.
